<a href="https://colab.research.google.com/github/darkvoid97/Mamba-AV2/blob/main/Mamba_AV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installing required libraries and Argoverse 2 Dataset extraction from Google Drive to local Colab memory

In [ ]:
!pip install uv
!uv pip install torch==2.9.0 torchaudio==2.9.0 torchvision==0.24.0 --index-url https://download.pytorch.org/whl/cu128
!uv pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
!uv pip install https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/av2_local/train
!unzip -q "PATH-TO-DATASET-FROM-GOOGLE-DRIVE" -d "/content/av2_local/train"
print("Done extracting the dataset on Colab's local memory!")

# Architecture and Dataset definition

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from pathlib import Path
from mamba_ssm import Mamba

# Mamba Model Architecture
class MultimodalMambaForecaster(nn.Module):
    def __init__(self, input_dim=4, d_model=256, n_layers=6, future_steps=60, out_dim=2, K=6):
        super().__init__()
        self.future_steps = future_steps
        self.out_dim = out_dim
        self.K = K

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model)
        )

        self.mamba_layers = nn.ModuleList([
            Mamba(d_model=d_model, d_state=16, d_conv=4, expand=2)
            for _ in range(n_layers)
        ])

        self.shared_head = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(0.1)
        )

        self.traj_head = nn.Linear(d_model * 2, K * future_steps * out_dim)
        self.prob_head = nn.Linear(d_model * 2, K)

    def forward(self, x):
        B = x.shape[0]
        hidden = self.input_proj(x)

        for layer in self.mamba_layers:
            hidden = layer(hidden)

        last_state = hidden[:, -1, :]
        shared_features = self.shared_head(last_state)

        trajectories = self.traj_head(shared_features)
        trajectories = trajectories.view(B, self.K, self.future_steps, self.out_dim)
        prob_logits = self.prob_head(shared_features)

        return trajectories, prob_logits

# Argoverse 2 Dataset
class RealArgoverse2Dataset(Dataset):
    def __init__(self, data_dir: str):
        print(f"Scansione dei file Parquet in {data_dir}...")
        self.scenario_files = list(Path(data_dir).rglob("*.parquet"))
        print(f"Trovati {len(self.scenario_files)} scenari.")
        self.hist_steps = 50
        self.fut_steps = 60

    def __len__(self):
        return len(self.scenario_files)

    def __getitem__(self, idx):
        file_path = self.scenario_files[idx]

        try:
            df = pd.read_parquet(file_path)

            focal_id = df['focal_track_id'].iloc[0]
            focal_df = df[df['track_id'] == focal_id].sort_values('timestep')

            if len(focal_df) != (self.hist_steps + self.fut_steps):
                raise ValueError("Scenario incompleto")

            pos_x = focal_df['position_x'].values
            pos_y = focal_df['position_y'].values
            vel_x = focal_df['velocity_x'].values
            vel_y = focal_df['velocity_y'].values

            # Normalization
            idx_origin = self.hist_steps - 1
            origin_x, origin_y = pos_x[idx_origin], pos_y[idx_origin]
            origin_yaw = np.arctan2(vel_y[idx_origin], vel_x[idx_origin])

            rel_x, rel_y = pos_x - origin_x, pos_y - origin_y
            cos_yaw, sin_yaw = np.cos(-origin_yaw), np.sin(-origin_yaw)
            rot_x = rel_x * cos_yaw - rel_y * sin_yaw
            rot_y = rel_x * sin_yaw + rel_y * cos_yaw
            rot_vx = vel_x * cos_yaw - vel_y * sin_yaw
            rot_vy = vel_x * sin_yaw + vel_y * cos_yaw

            # Neural Normalization
            # Dividing the meters by 50 and the velocity by 10
            # Getting them into the ideal range for the neural network
            rot_x = rot_x / 50.0
            rot_y = rot_y / 50.0
            rot_vx = rot_vx / 10.0
            rot_vy = rot_vy / 10.0

            features = np.stack([rot_x, rot_y, rot_vx, rot_vy], axis=-1)

            # Applying a filter at the source
            # In order to avoid any GPS or LiDAR sensors' glitches
            features = np.clip(features, a_min=-10.0, a_max=10.0)

            # Avoid NaN or Inf values corruption
            if np.isnan(features).any() or np.isinf(features).any():
                raise ValueError("Found NaN/Inf values in the scenario")

            return {
                'history': torch.tensor(features[:self.hist_steps], dtype=torch.float32),
                'future_gt': torch.tensor(features[self.hist_steps:, :2], dtype=torch.float32),
                'scenario_id': focal_df['scenario_id'].iloc[0]
            }

        except Exception as e:
            import random
            random_idx = random.randint(0, len(self.scenario_files) - 1)
            return self.__getitem__(random_idx)

    # Security fallback for corrupted scenarios
    def _get_empty_tensor(self):
        return {
            'history': torch.zeros((self.hist_steps, 4), dtype=torch.float32),
            'future_gt': torch.zeros((self.fut_steps, 2), dtype=torch.float32),
            'scenario_id': "CORRUPTED"
        }

# Winner-Takes-All Loss calculation done safely
def wta_loss(trajs_pred, logits_pred, y_true):
    B = trajs_pred.shape[0]
    y_true_exp = y_true.unsqueeze(1)

    with torch.no_grad():
        diff = trajs_pred[:, :, -1, :] - y_true_exp[:, :, -1, :]
        distances = torch.sqrt(torch.sum(diff**2, dim=-1) + 1e-6)
        best_k_indices = torch.argmin(distances, dim=1)

    best_trajs = trajs_pred[torch.arange(B), best_k_indices]

    reg_loss = F.smooth_l1_loss(best_trajs, y_true)
    cls_loss = F.cross_entropy(logits_pred, best_k_indices)

    return reg_loss + cls_loss

# minADE e minFDE calculation
def compute_metrics(trajs_pred, gt_traj):
    B = trajs_pred.shape[0]
    gt_traj_exp = gt_traj.unsqueeze(1)

    distances = torch.norm(trajs_pred - gt_traj_exp, dim=-1)
    fde_all = distances[:, :, -1]
    best_k_indices = torch.argmin(fde_all, dim=1)

    # Important: multiply back by 50 the values so that they'll be the actual ones
    min_fde = fde_all[torch.arange(B), best_k_indices] * 50.0
    min_ade = torch.mean(distances[torch.arange(B), best_k_indices], dim=1) * 50.0

    return min_ade.mean().item(), min_fde.mean().item()

# Training

In [ ]:
# Main training loop
if __name__ == "__main__":
    # Edit DATA_DIR for your own local path for the dataset
    DATA_DIR = "PATH-TO-LOCAL-DATASET"
    BATCH_SIZE = 64
    EPOCHS = 20
    LEARNING_RATE = 5e-5

    # Options for resuming training
    RESUME_TRAINING = False # ONLY put 'True' if you have a completed .pth file (check below)
    CHECKPOINT_PATH = "PATH-TO-LAST-PTH-FILE"
    START_EPOCH = 0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Beginning Training on: {device}")

    model = MultimodalMambaForecaster().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    # In case of resuming training: uploading the model and optimization
    if RESUME_TRAINING:
        print(f"Uploading completed checkpoint from {CHECKPOINT_PATH}...")
        # weights_only=False is necessary for uploading the optimizer's dictionaries
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        START_EPOCH = checkpoint['epoch']
        print(f"Successfully recovered weights and optimizer! Resuming training from epoch {START_EPOCH + 1}.")

    dataset = RealArgoverse2Dataset(data_dir=DATA_DIR)

    # num_workers > 0 is NECESSARY in order to read the parquet files without bottlenecking the GPU
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, drop_last=True)

    # Epochs loop
    for epoch in range(START_EPOCH, EPOCHS):
        model.train()
        epoch_loss = 0.0

        for batch_idx, batch in enumerate(dataloader):
            if batch['scenario_id'][0] == "CORRUPTED":
                continue

            x = batch['history'].to(device)
            y_true = batch['future_gt'].to(device)

            optimizer.zero_grad()
            trajs_pred, logits_pred = model(x)

            loss = wta_loss(trajs_pred, logits_pred, y_true)
            # Check if loss is acceptable.
            # Mamba models tend to 'explode' sometimes.
            if not torch.isfinite(loss):
                print(f"Anomalous loss at batch {batch_idx}! Skipping.")
                optimizer.zero_grad()
                continue

            if loss.item() > 20.0:
                print(f"Avoided anomalous peak (Loss: {loss.item():.2f}) at batch {batch_idx}. Skipping.")
                optimizer.zero_grad()
                continue

            # Calculate gradients
            loss.backward()

            # Check for gradients and find out if some of them 'exploded' before updating the weights
            has_bad_grad = False
            for param in model.parameters():
                if param.grad is not None and not torch.isfinite(param.grad).all():
                    has_bad_grad = True
                    break

            if has_bad_grad:
                print(f"Anomalous gradient found at batch {batch_idx}! Protecting the weights and skipping.")
                optimizer.zero_grad()
                continue

            # After checking, Gradient Clipping and Step
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()
            epoch_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch [{epoch+1}/{EPOCHS}] | Batch [{batch_idx}/{len(dataloader)}] | Loss: {loss.item():.4f}")

        scheduler.step()

        # Eval after completing the epoch
        model.eval()
        val_ade, val_fde = 0.0, 0.0
        val_batches = 0

        print("Evaluating metrics...")
        with torch.no_grad():
            for i, batch in enumerate(dataloader):
                if i >= 20: break
                if batch['scenario_id'][0] == "CORRUPTED": continue

                x = batch['history'].to(device)
                y_true = batch['future_gt'].to(device)
                trajs_pred, _ = model(x)

                ade, fde = compute_metrics(trajs_pred, y_true)

                # Protection while evaluating
                import math
                if not math.isinf(ade) and not math.isnan(ade):
                    val_ade += ade
                    val_fde += fde
                    val_batches += 1
                else:
                    print(f"  [Warning] Skipped anomalous evaluating batch ({i})")

        print(f"--- END EPOCH {epoch+1} ---")
        # In case too many batches have been removed
        if val_batches > 0:
            print(f"minADE: {val_ade/val_batches:.4f} m | minFDE: {val_fde/val_batches:.4f} m")
        else:
            print("No valid evaluating batch found in this epoch.")
        print("-" * 30)

        # Epoch Checkpoint save
        checkpoint_name = f"mamba_av2_epoch_{epoch+1}.pth"
        checkpoint_data = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': epoch_loss / len(dataloader)
        }
        torch.save(checkpoint_data, checkpoint_name)
        print(f"Saved completed checkpoint in: {checkpoint_name}")
        print("-" * 30)

# Model Inference

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import plotly.graph_objects as go

# Get a scenario, make the inference and draw the result in an interactive graphics
def visualize_prediction(model, dataset, scenario_idx=None, device='cuda'):
    model.eval()

    if scenario_idx is None:
        scenario_idx = np.random.randint(0, len(dataset))

    print(f"Visualization Scenario Index: {scenario_idx}")

    data = dataset[scenario_idx]
    if data['scenario_id'] == "CORRUPTED":
        return visualize_prediction(model, dataset, device=device)

    history = data['history'].unsqueeze(0).to(device)
    gt_future = data['future_gt'].unsqueeze(0).to(device)

    with torch.no_grad():
        trajs_pred, logits_pred = model(history)
        probs = torch.nn.functional.softmax(logits_pred, dim=-1)[0]

    history_np = history[0].cpu().numpy()
    gt_future_np = gt_future[0].cpu().numpy()
    trajs_pred_np = trajs_pred[0].cpu().numpy()
    probs_np = probs.cpu().numpy()

    # Denormalization in actual meters
    history_np[:, :2] = history_np[:, :2] * 50.0
    gt_future_np = gt_future_np * 50.0
    trajs_pred_np = trajs_pred_np * 50.0

    # Plotly part for the graphics
    fig = go.Figure()

    # Origin point (Vehicle in T=0)
    fig.add_trace(go.Scatter(x=[0], y=[0], mode='markers', name='Current Position',
                             marker=dict(symbol='star', size=16, color='black'),
                             hoverinfo='name'))

    # History (past moments)
    fig.add_trace(go.Scatter(x=history_np[:, 0], y=history_np[:, 1],
                             mode='lines+markers', name='History (Past 5s)',
                             line=dict(color='blue', width=2),
                             marker=dict(size=4, opacity=0.5)))

    # Ground Truth (actual future)
    fig.add_trace(go.Scatter(x=gt_future_np[:, 0], y=gt_future_np[:, 1],
                             mode='lines', name='Ground Truth (Real)',
                             line=dict(color='green', width=4, dash='dash')))

    # Mamba model predictions
    best_idx = np.argmax(probs_np)

    for k in range(model.K):
        traj = trajs_pred_np[k]
        prob = probs_np[k] * 100

        if k == best_idx:
            fig.add_trace(go.Scatter(x=traj[:, 0], y=traj[:, 1],
                                     mode='lines', name=f'Best Pred ({prob:.1f}%)',
                                     line=dict(color='red', width=4)))
        else:
            alpha_val = max(0.3, probs_np[k])
            fig.add_trace(go.Scatter(x=traj[:, 0], y=traj[:, 1],
                                     mode='lines', name=f'Alt Mode ({prob:.1f}%)',
                                     line=dict(color='orange', width=2),
                                     opacity=alpha_val))

    # Layout
    fig.update_layout(
        title=f"Mamba HD Prediction - Scenario: {data['scenario_id']}",
        xaxis_title="X Coordinate (Meters)",
        yaxis_title="Y Coordinate (Meters)",
        yaxis=dict(scaleanchor="x", scaleratio=1),
        width=1000,
        height=800,
        template="plotly_white",
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01) # Dictionary on the top left
    )

    fig.show()

# Script Execution
if __name__ == "__main__":
    DATA_DIR = "PATH-TO-LOCAL-DATASET"
    CHECKPOINT_PATH = "PATH-TO-LAST-PTH-FILE"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Dataset initialization
    dataset = RealArgoverse2Dataset(data_dir=DATA_DIR)

    # Modello initialization and weights loading
    model = MultimodalMambaForecaster().to(device)
    print(f"Loading model from {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

    # Check if the entire dictionary was saves, or just the weights
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)

    print("Model successfully initialized!")

    # Keep executing this cell to visualize different scenarios
    visualize_prediction(model, dataset, device=device)

# Metrics calculation (minADE e minFDE)

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader

# Evaluate minADE, minFDE and Miss Rate.
# limit_batches: if set to a number (example: 500), it just checks for the first N batches
def evaluate_model(model, dataloader, device='cuda', limit_batches=None):
    model.eval()

    total_ade = 0.0
    total_fde = 0.0
    total_misses = 0
    total_samples = 0

    print("Beginning evaluation...")

    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if limit_batches is not None and batch_idx >= limit_batches:
                break

            if batch['scenario_id'][0] == "CORRUPTED":
                continue

            history = batch['history'].to(device)
            gt_future = batch['future_gt'].to(device)

            trajs_pred, logits_pred = model(history)

            # Denormalization to get actual meters
            trajs_pred = trajs_pred * 50.0
            gt_future = gt_future * 50.0

            B, K, T, _ = trajs_pred.shape

            # Evaluating metrics for every scenario in the batch
            for b in range(B):
                gt = gt_future[b]
                preds = trajs_pred[b]

                # Get the FDE for all 6 of the predicted trajectories
                final_dists = torch.norm(preds[:, -1, :] - gt[-1, :], dim=-1)

                # Get the best trajectory
                best_mode_idx = torch.argmin(final_dists)
                best_traj = preds[best_mode_idx]

                # Calculate the minADE and minFDE on that best trajectory
                fde = final_dists[best_mode_idx].item()
                ade = torch.mean(torch.norm(best_traj - gt, dim=-1)).item()

                # Miss Rate (for Argoverse it's > 2 meters)
                is_miss = 1 if fde > 2.0 else 0

                # Update the totals
                total_ade += ade
                total_fde += fde
                total_misses += is_miss
                total_samples += 1

            if (batch_idx + 1) % 100 == 0:
                print(f"{total_samples} scenarios processed...")

    # Calcolo Medie Finali
    mean_ade = total_ade / total_samples
    mean_fde = total_fde / total_samples
    miss_rate = (total_misses / total_samples) * 100

    print("\n" + "="*30)
    print("EVALUATION FINAL RESULTS")
    print("="*30)
    print(f"Number of scenarios : {total_samples}")
    print(f"minADE           : {mean_ade:.4f} meters")
    print(f"minFDE           : {mean_fde:.4f} meters")
    print(f"Miss Rate (> 2m) : {miss_rate:.2f} %")
    print("="*30)

    return mean_ade, mean_fde, miss_rate

# Starting script
if __name__ == "__main__":
    DATA_DIR = "PATH-TO-LOCAL-DATASET"
    CHECKPOINT_PATH = "PATH-TO-LAST-PTH-FILE"
    BATCH_SIZE = 64

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Data loading (shuffle=False is better in the case of evaluation)
    dataset = RealArgoverse2Dataset(data_dir=DATA_DIR)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    # Model initialization
    model = MultimodalMambaForecaster().to(device)

    print(f"Loading model from {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)

    # Evaluate model
    # Edit limit_batches=100 if you only want a fast preview on ~6400 scenarios
    evaluate_model(model, dataloader, device=device, limit_batches=1000)